In [ ]:
!pip install gradio transformers torch tf-keras datasets

In [ ]:
import gradio as gr
from transformers import pipeline
from datasets import load_dataset

print("Loading SQuAD dataset...")
# Load SQuAD v1.1 dataset
squad_dataset = load_dataset("squad", split="train")
print(f"Loaded {len(squad_dataset)} examples from SQuAD dataset")

# Extract first 10 examples (context, question, and answer)
squad_examples = []
for i in range(10):
    example = squad_dataset[i]
    # Get the answer text (first answer if multiple exist)
    answer_text = example['answers']['text'][0] if example['answers']['text'] else ""
    squad_examples.append([
        example['context'],
        example['question'],
        answer_text  # Ground truth answer from dataset
    ])
print(f"Extracted first {len(squad_examples)} examples with answers")

print("Loading models...")

# --- Model 1: DistilBERT (Fast, SQuAD v1.1) ---
# Good for speed, trained on answerable questions only.
model_1_name = "distilbert-base-cased-distilled-squad"
qa_pipeline_1 = pipeline("question-answering", model=model_1_name)

# --- Model 2: RoBERTa (Accurate, SQuAD v2.0) ---
# Good for accuracy, can handle "unanswerable" questions (returns empty string).
model_2_name = "deepset/roberta-base-squad2"
qa_pipeline_2 = pipeline("question-answering", model=model_2_name)

def compare_models(context, question):
    # Error handling for empty inputs
    if not context or not question:
        return "N/A", 0.0, "N/A", 0.0

    # Run Model 1
    res1 = qa_pipeline_1(question=question, context=context)
    
    # Run Model 2
    res2 = qa_pipeline_2(question=question, context=context)

    # Return: Ans1, Conf1, Ans2, Conf2
    return res1['answer'], res1['score'], res2['answer'], res2['score']

# --- Gradio UI Layout ---
with gr.Blocks(title="QA Model Arena") as demo:
    gr.Markdown("# ⚔️ QA Model Arena: DistilBERT vs. RoBERTa")
    gr.Markdown("Compare a lightweight model (DistilBERT) against a robust model (RoBERTa) on the **SQuAD** dataset.")
    
    with gr.Row():
        with gr.Column(scale=1):
            context_input = gr.Textbox(lines=8, label="Context Paragraph", placeholder="Paste context here...")
            question_input = gr.Textbox(lines=2, label="Question")
            ground_truth_answer = gr.Textbox(lines=2, label="Ground Truth Answer (from SQuAD)", interactive=False)
            submit_btn = gr.Button("Compare Models", variant="primary")
        
        with gr.Column(scale=1):
            gr.Markdown(f"### 🚀 Model A: {model_1_name}")
            out_ans_1 = gr.Textbox(label="Answer")
            out_conf_1 = gr.Number(label="Confidence")
            
            gr.Markdown("---")
            
            gr.Markdown(f"### 🧠 Model B: {model_2_name}")
            out_ans_2 = gr.Textbox(label="Answer")
            out_conf_2 = gr.Number(label="Confidence")

    # Link inputs/outputs
    submit_btn.click(
        fn=compare_models,
        inputs=[context_input, question_input],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2]
    )

    # Add Examples
    gr.Examples(
        examples=squad_examples,
        inputs=[context_input, question_input, ground_truth_answer],
        outputs=[out_ans_1, out_conf_1, out_ans_2, out_conf_2],
        fn=compare_models,
        cache_examples=True,
    )

if __name__ == "__main__":
    demo.launch()